In [ ]:
### Imports

import pathlib

import h5py
import matplotlib.pyplot as plt
import numpy as np
import polars as pls

import usv_playpen.analyses.neuronal_coactivity_engine as engine

In [ ]:
### Load data 

complex_cat_ids = [1, 2]
simple_cat_ids = [5]

# List of session directories to process
# data_directories = [
#     pathlib.Path('/mnt/falkner/Bartul/Data/20250923_162249'),
#     pathlib.Path('/mnt/falkner/Bartul/Data/20250923_171803'),
#     pathlib.Path('/mnt/falkner/Bartul/Data/20250923_174642'),
#     pathlib.Path('/mnt/falkner/Bartul/Data/20250923_184326'),
#     pathlib.Path('/mnt/falkner/Bartul/Data/20250923_200538'),
#     pathlib.Path('/mnt/falkner/Bartul/Data/20250923_203320')
# ]

# data_directories = [
#     pathlib.Path('/mnt/falkner/Bartul/Data/20250919_152924'),
#     pathlib.Path('/mnt/falkner/Bartul/Data/20250919_163351'),
# ]


data_directories = [
    pathlib.Path('/mnt/falkner/Bartul/Data/20250927_142335'),
    pathlib.Path('/mnt/falkner/Bartul/Data/20250927_151825'),
    pathlib.Path('/mnt/falkner/Bartul/Data/20250927_172337'),
    pathlib.Path('/mnt/falkner/Bartul/Data/20250927_181936'),
]

def get_common_units(directories: list[pathlib.Path]) -> set[str]:
    """
    Identifies the intersection of 'good' neural units across all sessions.

    Parameters
    ----------
    directories : list[pathlib.Path]
        List of session paths to search.

    Returns
    -------
    set[str]
        A set of unit names (stems) that exist in every directory.
    """
    unit_sets = []
    for directory in directories:
        # Extract stems for all files ending in _good.npy
        units = {f.stem for f in directory.glob('**/*_good.npy')}
        unit_sets.append(units)
    
    if not unit_sets:
        return set()
    
    # Return only units found in every single session directory
    return set.intersection(*unit_sets)

# 1. Identify consistent units across all sessions
common_unit_ids = get_common_units(data_directories)
print(f"Found {len(common_unit_ids)} units present across all {len(data_directories)} sessions.")

# 2. Data Containers (List of dictionaries for each session)
sessions_data = []

for directory in data_directories:
    print(f"Processing session: {directory.name}")
    
    # --- Tracking Data ---
    tracking_file = next(directory.glob('**/*_translated_rotated_metric.h5'))
    with h5py.File(name=tracking_file, mode='r') as track_file:
        mouse_tracks = np.array(track_file['tracks'])
        mouse_track_names = [item.decode('utf-8') for item in list(track_file['track_names'])]
        recording_frame_rate = float(track_file['recording_frame_rate'][()])

    # --- Vocalization Data ---
    usv_summary_file = next(directory.glob('**/*_usv_summary.csv'))
    usv_summary_data = pls.read_csv(usv_summary_file)
    
    # Filter by emitter (male) and create sub-category dataframes using hyperparameters
    male_usvs = usv_summary_data.filter(pls.col('emitter') == mouse_track_names[0])
    
    complex_df = male_usvs.filter(pls.col('usv_supercategory').is_in(complex_cat_ids))
    simple_df = male_usvs.filter(pls.col('usv_supercategory').is_in(simple_cat_ids))

    # --- Neural Data (Only Common Units) ---
    session_neural_data = {}
    for unit_id in common_unit_ids:
        # Construct path to the unit file within this session
        spike_file = next(directory.glob(f'**/{unit_id}.npy'))
        # Load and extract the spike times (second dimension per your original code)
        session_neural_data[unit_id] = np.load(spike_file)[0, :]

    # --- Store Session Payload ---
    sessions_data.append({
        'session_id': directory.name,
        'tracks': mouse_tracks,
        'fs': recording_frame_rate,
        'complex_df': complex_df,
        'simple_df': simple_df,
        'neural_data': session_neural_data,
        'total_duration': mouse_tracks.shape[0] / recording_frame_rate
    })

print(f"\nSuccessfully loaded {len(sessions_data)} sessions for chained analysis.")

In [ ]:
### Compute Coactivity Metrics for Complex vs. Simple Vocalizations

usv_bootstrap_num = 300    
n_boot_iterations = 1000
n_shuffles = 1000
WINDOW_S = 0.030 

# --- 1. Aggregate Observed Count Matrices ---
complex_mats = []
simple_mats = []

for sess in sessions_data:
    c_mat = engine.extract_snippet_matrix(sess['complex_df']['start'].to_numpy(), sess['neural_data'], WINDOW_S)
    s_mat = engine.extract_snippet_matrix(sess['simple_df']['start'].to_numpy(), sess['neural_data'], WINDOW_S)
    complex_mats.append(c_mat)
    simple_mats.append(s_mat)

# Concatenate trials (axis=1) across all sessions
all_complex_counts = np.hstack(complex_mats)
all_simple_counts = np.hstack(simple_mats)

print(f"Aggregated Data: {all_complex_counts.shape[1]} Complex USVs | {all_simple_counts.shape[1]} Simple USVs")

# --- 2. Bootstrapping (Now on Pooled Data) ---
# Use the same matched N logic across the whole dataset
all_complex_boot = engine.get_bootstrapped_simple_calls(all_complex_counts, usv_bootstrap_num, n_boot_iterations)
all_simple_boot = engine.get_bootstrapped_simple_calls(all_simple_counts, usv_bootstrap_num, n_boot_iterations)

# --- 3. Aggregate Observed Count Matrices ---
complex_mats = []
simple_mats = []

for sess in sessions_data:
    c_mat = engine.extract_snippet_matrix(sess['complex_df']['start'].to_numpy(), sess['neural_data'], WINDOW_S)
    s_mat = engine.extract_snippet_matrix(sess['simple_df']['start'].to_numpy(), sess['neural_data'], WINDOW_S)
    complex_mats.append(c_mat)
    simple_mats.append(s_mat)

all_complex_counts = np.hstack(complex_mats)
all_simple_counts = np.hstack(simple_mats)

print(f"Aggregated Data: {all_complex_counts.shape[1]} Complex trials | {all_simple_counts.shape[1]} Simple trials")

# --- 4. Chained Bootstrapping (N=matched) ---
print(f"Bootstrapping pooled data to N={usv_bootstrap_num}...")
all_complex_boot = engine.get_bootstrapped_simple_calls(all_complex_counts, usv_bootstrap_num, n_boot_iterations)
all_simple_boot = engine.get_bootstrapped_simple_calls(all_simple_counts, usv_bootstrap_num, n_boot_iterations)

# --- 5. Chained Circular Shuffle (N=matched) ---
print(f"Generating matched-N chained null distributions...")

# Use the robust sampling function to get timing templates
session_complex_onsets = engine.sample_onsets_across_sessions(sessions_data, 'complex_df', usv_bootstrap_num)
session_simple_onsets = engine.sample_onsets_across_sessions(sessions_data, 'simple_df', usv_bootstrap_num)

# Execute the chained shuffle using the lists of session data
complex_chained_null = engine.perform_chained_circular_shuffle(
    session_onsets=session_complex_onsets,
    session_neural_data=[sess['neural_data'] for sess in sessions_data],
    session_durations=[sess['total_duration'] for sess in sessions_data],
    window_s=WINDOW_S,
    n_shuffles=n_shuffles
)

simple_chained_null = engine.perform_chained_circular_shuffle(
    session_onsets=session_simple_onsets,
    session_neural_data=[sess['neural_data'] for sess in sessions_data],
    session_durations=[sess['total_duration'] for sess in sessions_data],
    window_s=WINDOW_S,
    n_shuffles=n_shuffles
)

# --- 6. Statistical Validation ---

def calculate_stats(boot_data, null_data):
    """
    Compares the mean of the bootstrap distribution to the 
    shuffled null distribution to derive empirical p and Z.
    """
    boot_mean = np.mean(boot_data)
    null_mean = np.mean(null_data)
    null_std = np.std(null_data)
    
    # Calculate empirical p-value (right-tailed)
    p_val = np.mean(null_data >= boot_mean)
    
    # Calculate Z-score relative to the null distribution
    z_score = (boot_mean - null_mean) / null_std if null_std > 0 else 0
    
    return boot_mean, null_mean, p_val, z_score

# Compute stats for COMPLEX (using pooled bootstrap and chained null)
c_rsc_obs, c_rsc_null, c_rsc_p, c_rsc_z = calculate_stats(all_complex_boot['r_sc'], complex_chained_null['r_sc'])
c_sim_obs, c_sim_null, c_sim_p, c_sim_z = calculate_stats(all_complex_boot['similarity'], complex_chained_null['similarity'])
c_pop_obs, c_pop_null, c_pop_p, c_pop_z = calculate_stats(all_complex_boot['pop_corr'], complex_chained_null['pop_corr'])

# Compute stats for SIMPLE (using pooled bootstrap and chained null)
s_rsc_obs, s_rsc_null, s_rsc_p, s_rsc_z = calculate_stats(all_simple_boot['r_sc'], simple_chained_null['r_sc'])
s_sim_obs, s_sim_null, s_sim_p, s_sim_z = calculate_stats(all_simple_boot['similarity'], simple_chained_null['similarity'])
s_pop_obs, s_pop_null, s_pop_p, s_pop_z = calculate_stats(all_simple_boot['pop_corr'], simple_chained_null['pop_corr'])

# --- 6. Display Summary Results ---

print("-" * 75)
print(f"SYMMETRIC CHAINED COMPARISON (Matched N={usv_bootstrap_num} across {len(sessions_data)} sessions)")
print("-" * 75)

print(f"COMPLEX (Resampled N={usv_bootstrap_num}):")
print(f"  r_sc:       Boot={c_rsc_obs:.4f} | Null={c_rsc_null:.4f} | p={c_rsc_p:.4f} (Z={c_rsc_z:.2f})")
print(f"  Similarity: Boot={c_sim_obs:.4f} | Null={c_sim_null:.4f} | p={c_sim_p:.4f} (Z={c_sim_z:.2f})")
print(f"  Pop Corr:   Boot={c_pop_obs:.4f} | Null={c_pop_null:.4f} | p={c_pop_p:.4f} (Z={c_pop_z:.2f})")

print("-" * 75)

print(f"SIMPLE (Resampled N={usv_bootstrap_num}):")
print(f"  r_sc:       Boot={s_rsc_obs:.4f} | Null={s_rsc_null:.4f} | p={s_rsc_p:.4f} (Z={s_rsc_z:.2f})")
print(f"  Similarity: Boot={s_sim_obs:.4f} | Null={s_sim_null:.4f} | p={s_sim_p:.4f} (Z={s_sim_z:.2f})")
print(f"  Pop Corr:   Boot={s_pop_obs:.4f} | Null={s_pop_null:.4f} | p={s_pop_p:.4f} (Z={s_pop_z:.2f})")

In [ ]:

# 1. Calculate category-specific thresholds (99th Percentile)
# Using the chained null distributions (the global yardstick)
c_rsc_99 = np.percentile(complex_chained_null['r_sc'], 99)
c_sim_99 = np.percentile(complex_chained_null['similarity'], 99)
c_pop_99 = np.percentile(complex_chained_null['pop_corr'], 99)

s_rsc_99 = np.percentile(simple_chained_null['r_sc'], 99)
s_sim_99 = np.percentile(simple_chained_null['similarity'], 99)
s_pop_99 = np.percentile(simple_chained_null['pop_corr'], 99)

# 2. Extract Pooled Bootstrap Means (Aggregated across all sessions)
c_val_rsc = np.mean(all_complex_boot['r_sc'])
c_val_sim = np.mean(all_complex_boot['similarity'])
c_val_pop = np.mean(all_complex_boot['pop_corr'])

s_val_rsc = np.mean(all_simple_boot['r_sc'])
s_val_sim = np.mean(all_simple_boot['similarity'])
s_val_pop = np.mean(all_simple_boot['pop_corr'])

# 3. Plotting: 3 Rows (Metric Type) x 2 Columns (Call Type)
fig, axes = plt.subplots(3, 2, figsize=(14, 15), sharey='row')

# --- ROW 1: Pairwise Correlation (r_sc) ---
# Complex
ax = axes[0, 0]
ax.hist(complex_chained_null['r_sc'], bins=40, color='grey', alpha=0.5, label='Chained Null')
ax.axvline(c_rsc_99, color='black', linestyle='--', label='99%')
ax.axvline(c_val_rsc, color='crimson', linewidth=3, label=f'Pooled Boot ({c_val_rsc:.4f})')
ax.set_title('COMPLEX: Pairwise Correlation ($r_{sc}$)', fontsize=14)
ax.legend(fontsize=8)

# Simple
ax = axes[0, 1]
ax.hist(simple_chained_null['r_sc'], bins=40, color='grey', alpha=0.5, label='Chained Null')
ax.axvline(s_rsc_99, color='black', linestyle='--', label='99%')
ax.axvline(s_val_rsc, color='dodgerblue', linewidth=3, label=f'Pooled Boot ({s_val_rsc:.4f})')
ax.set_title('SIMPLE: Pairwise Correlation ($r_{sc}$)', fontsize=14)
ax.legend(fontsize=8)

# --- ROW 2: Population Similarity (Cosine) ---
# Complex
ax = axes[1, 0]
ax.hist(complex_chained_null['similarity'], bins=40, color='grey', alpha=0.5, label='Chained Null')
ax.axvline(c_sim_99, color='black', linestyle='--', label='99%')
ax.axvline(c_val_sim, color='crimson', linewidth=3, label=f'Pooled Boot ({c_val_sim:.4f})')
ax.set_title('COMPLEX: Cosine Similarity', fontsize=14)
ax.legend(fontsize=8)

# Simple
ax = axes[1, 1]
ax.hist(simple_chained_null['similarity'], bins=40, color='grey', alpha=0.5, label='Chained Null')
ax.axvline(s_sim_99, color='black', linestyle='--', label='99%')
ax.axvline(s_val_sim, color='dodgerblue', linewidth=3, label=f'Pooled Boot ({s_val_sim:.4f})')
ax.set_title('SIMPLE: Cosine Similarity', fontsize=14)
ax.legend(fontsize=8)

# --- ROW 3: Population Vector Correlation (Pearson) ---
# Complex
ax = axes[2, 0]
ax.hist(complex_chained_null['pop_corr'], bins=40, color='grey', alpha=0.5, label='Chained Null')
ax.axvline(c_pop_99, color='black', linestyle='--', label='99%')
ax.axvline(c_val_pop, color='crimson', linewidth=3, label=f'Pooled Boot ({c_val_pop:.4f})')
ax.set_title('COMPLEX: Pop Vector Corr (Pearson)', fontsize=14)
ax.set_xlabel('Mean Pearson Correlation', fontsize=12)
ax.legend(fontsize=8)

# Simple
ax = axes[2, 1]
ax.hist(simple_chained_null['pop_corr'], bins=40, color='grey', alpha=0.5, label='Chained Null')
ax.axvline(s_pop_99, color='black', linestyle='--', label='99%')
ax.axvline(s_val_pop, color='dodgerblue', linewidth=3, label=f'Pooled Boot ({s_val_pop:.4f})')
ax.set_title('SIMPLE: Pop Vector Corr (Pearson)', fontsize=14)
ax.set_xlabel('Mean Pearson Correlation', fontsize=12)
ax.legend(fontsize=8)

# Cleanup spines and layout
for ax_row in axes:
    for ax in ax_row:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()